In [1]:
#Import libraries
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
#Load the raw dataset
df = pd.read_csv(
    r"C:\Users\AIRA\OneDrive\Desktop\241BCADA06\Desktop\ML PROJECT\workspace\credit_card_fraud.csv"
)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully.
Rows: 339607
Columns: 15


In [5]:
#Check the dataset
print("Column names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nDataset information:")
df.info()

Column names:
['trans_date_trans_time', 'merchant', 'category', 'amt', 'city', 'state', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'merch_lat', 'merch_long', 'is_fraud']

First 5 rows:


,trans_date_trans_time,merchant,category,amt,city,state,lat,long,city_pop,job,dob,trans_num,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:44,"Heller, Gutmann and Zieme",grocery_pos,107.23,Orient,WA,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,49.159047,-118.186462,0
1,2019-01-01 00:00:51,Lind-Buckridge,entertainment,220.11,Malad City,ID,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,43.150704,-112.154481,0
2,2019-01-01 00:07:27,Kiehn Inc,grocery_pos,96.29,Grenada,CA,41.6125,-122.5258,589,Systems analyst,1945-12-21,413636e759663f264aae1819a4d4f231,41.657520,-122.230347,0
3,2019-01-01 00:09:03,Beier-Hyatt,shopping_pos,7.77,High Rolls Mountain Park,NM,32.9396,-105.8189,899,Naval architect,1967-08-30,8a6293af5ed278dea14448ded2685fea,32.863258,-106.520205,0
4,2019-01-01 00:21:32,Bruen-Yost,misc_pos,6.85,Freedom,WY,43.0172,-111.0292,471,"Education officer, museum",1967-08-02,f3c43d336e92a44fc2fb67058d5949e3,43.753735,-111.454923,0



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 339607 entries, 0 to 339606
Data columns (total 15 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   trans_date_trans_time  339607 non-null  object 
 1   merchant               339607 non-null  object 
 2   category               339607 non-null  object 
 3   amt                    339607 non-null  float64
 4   city                   339607 non-null  object 
 5   state                  339607 non-null  object 
 6   lat                    339607 non-null  float64
 7   long                   339607 non-null  float64
 8   city_pop               339607 non-null  int64  
 9   job                    339607 non-null  object 
 10  dob                    339607 non-null  object 
 11  trans_num              339607 non-null  object 
 12  merch_lat              339607 non-null  float64
 13  merch_long             339607 non-null  float64
 14  is_fraud      

In [7]:
#Schema validation
expected_columns = [
    "trans_date_trans_time",
    "merchant",
    "category",
    "amt",
    "city",
    "state",
    "lat",
    "long",
    "city_pop",
    "job",
    "dob",
    "trans_num",
    "merch_lat",
    "merch_long",
    "is_fraud"
]

missing_columns = [
    col for col in expected_columns
    if col not in df.columns
]

extra_columns = [
    col for col in df.columns
    if col not in expected_columns
]

print("Missing columns:", missing_columns)
print("Extra columns:", extra_columns)

if len(missing_columns) == 0 and len(extra_columns) == 0:
    print("Schema validation PASSED")
else:
    print("Schema validation FAILED")

Missing columns: []
Extra columns: []
Schema validation PASSED


In [9]:
#Check missing values
print("Missing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

Missing values in each column:
trans_date_trans_time    0
merchant                 0
category                 0
amt                      0
city                     0
state                    0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
merch_lat                0
merch_long               0
is_fraud                 0
dtype: int64

Total missing values: 0


In [11]:
#Check duplicate records
print("Duplicate rows:", df.duplicated().sum())

print(
    "Duplicate transaction IDs:",
    df["trans_num"].duplicated().sum()
)

Duplicate rows: 0
Duplicate transaction IDs: 0


In [13]:
#Check fraud labels
print("Fraud label values:")
print(df["is_fraud"].unique())

print("\nFraud label counts:")
print(df["is_fraud"].value_counts())

print("\nFraud label percentage:")
print(
    df["is_fraud"].value_counts(normalize=True) * 100
)

Fraud label values:
[0 1]

Fraud label counts:
is_fraud
0    337825
1      1782
Name: count, dtype: int64

Fraud label percentage:
is_fraud
0    99.475276
1     0.524724
Name: proportion, dtype: float64


In [15]:
#Validate numerical values
validation_report = {
    "Negative amount": (df["amt"] < 0).sum(),

    "Invalid latitude": (
        (df["lat"] < -90) |
        (df["lat"] > 90)
    ).sum(),

    "Invalid longitude": (
        (df["long"] < -180) |
        (df["long"] > 180)
    ).sum(),

    "Invalid merchant latitude": (
        (df["merch_lat"] < -90) |
        (df["merch_lat"] > 90)
    ).sum(),

    "Invalid merchant longitude": (
        (df["merch_long"] < -180) |
        (df["merch_long"] > 180)
    ).sum(),

    "Invalid fraud labels": (
        ~df["is_fraud"].isin([0, 1])
    ).sum()
}

print("Data validation report:")
display(pd.Series(validation_report))

Data validation report:


Negative amount               0
Invalid latitude              0
Invalid longitude             0
Invalid merchant latitude     0
Invalid merchant longitude    0
Invalid fraud labels          0
dtype: int64

In [17]:
#Convert date columns
df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"]
)

df["dob"] = pd.to_datetime(
    df["dob"]
)

print("Date conversion completed.")

Date conversion completed.


In [19]:
#Feature engineering
df["trans_hour"] = (
    df["trans_date_trans_time"].dt.hour
)

df["trans_day"] = (
    df["trans_date_trans_time"].dt.day
)

df["trans_month"] = (
    df["trans_date_trans_time"].dt.month
)

df["trans_dayofweek"] = (
    df["trans_date_trans_time"].dt.dayofweek
)

df["is_weekend"] = (
    df["trans_dayofweek"] >= 5
).astype(int)

df["age"] = (
    (df["trans_date_trans_time"] - df["dob"]).dt.days
    / 365.25
)

print("Feature engineering completed.")

Feature engineering completed.


In [21]:
#Check amount outliers
Q1 = df["amt"].quantile(0.25)
Q3 = df["amt"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

amount_outliers = (
    (df["amt"] < lower_limit) |
    (df["amt"] > upper_limit)
)

print("Amount outliers:", amount_outliers.sum())

print(
    "Outlier percentage:",
    round(amount_outliers.mean() * 100, 2),
    "%"
)

Amount outliers: 18043
Outlier percentage: 5.31 %


In [23]:
#Separate features and target
X = df.drop(
    columns=[
        "is_fraud",
        "trans_num",
        "trans_date_trans_time",
        "dob"
    ]
)

y = df["is_fraud"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (339607, 17)
y shape: (339607,)


In [25]:
#Define numerical and categorical features
numeric_features = [
    "amt",
    "lat",
    "long",
    "city_pop",
    "merch_lat",
    "merch_long",
    "trans_hour",
    "trans_day",
    "trans_month",
    "trans_dayofweek",
    "is_weekend",
    "age"
]

categorical_features = [
    "merchant",
    "category",
    "city",
    "state",
    "job"
]

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numerical features: 12
Categorical features: 5


In [27]:
#Split training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training data: (271685, 17)
Testing data: (67922, 17)
Training target: (271685,)
Testing target: (67922,)


In [29]:
#Numerical preprocessing
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

In [31]:
#Categorical preprocessing
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])

In [33]:
#Create the preprocessing pipeline
preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [35]:
#Process training data
X_train_processed = preprocessor.fit_transform(
    X_train
)

print(
    "Processed training data:",
    X_train_processed.shape
)

Processed training data: (271685, 1071)


In [37]:
#Process testing data
X_test_processed = preprocessor.transform(
    X_test
)

print(
    "Processed testing data:",
    X_test_processed.shape
)

Processed testing data: (67922, 1071)


In [39]:
#Validate processed data
print("========== PROCESSED DATA VALIDATION ==========")

print("Training rows:", X_train_processed.shape[0])
print("Testing rows:", X_test_processed.shape[0])

print(
    "Processed training features:",
    X_train_processed.shape[1]
)

print(
    "Processed testing features:",
    X_test_processed.shape[1]
)

assert X_train_processed.shape[0] == len(y_train)
assert X_test_processed.shape[0] == len(y_test)

assert (
    X_train_processed.shape[1]
    == X_test_processed.shape[1]
)

print("\nProcessed data validation PASSED!")

========== PROCESSED DATA VALIDATION ==========
Training rows: 271685
Testing rows: 67922
Processed training features: 1071
Processed testing features: 1071

Processed data validation PASSED!


In [41]:
#Get processed feature names
feature_names = (
    preprocessor.get_feature_names_out()
)

print(
    "Number of processed features:",
    len(feature_names)
)

print("\nFirst 10 processed features:")
print(feature_names[:10])

Number of processed features: 1071

First 10 processed features:
['numeric__amt' 'numeric__lat' 'numeric__long' 'numeric__city_pop'
 'numeric__merch_lat' 'numeric__merch_long' 'numeric__trans_hour'
 'numeric__trans_day' 'numeric__trans_month' 'numeric__trans_dayofweek']


In [43]:
#Create processed-data folder
processed_dir = "data/processed"

os.makedirs(
    processed_dir,
    exist_ok=True
)

print(
    "Processed data folder:",
    os.path.abspath(processed_dir)
)

Processed data folder: C:\Users\AIRA\data\processed


In [55]:
processed_df = df.drop(
    columns=[
        "trans_num",
        "trans_date_trans_time",
        "dob"
    ]
)

processed_file = os.path.join(
    processed_dir,
    "processed_credit_card_fraud.csv"
)

processed_df.to_csv(
    processed_file,
    index=False
)

print("Processed dataset saved successfully!")
print("File:", os.path.abspath(processed_file))
print("Shape:", processed_df.shape)

Processed dataset saved successfully!
File: C:\Users\AIRA\data\processed\processed_credit_card_fraud.csv
Shape: (339607, 18)


In [47]:
# Save training and testing data
X_train.to_csv(
    os.path.join(
        processed_dir,
        "X_train.csv"
    ),
    index=False
)

X_test.to_csv(
    os.path.join(
        processed_dir,
        "X_test.csv"
    ),
    index=False
)

y_train.to_csv(
    os.path.join(
        processed_dir,
        "y_train.csv"
    ),
    index=False
)

y_test.to_csv(
    os.path.join(
        processed_dir,
        "y_test.csv"
    ),
    index=False
)

print("Training and testing CSV files saved successfully!")

Training and testing CSV files saved successfully!


In [49]:
#Create the data quality audit log
quality_log = pd.DataFrame({
    "Check": [
        "Dataset rows",
        "Dataset columns",
        "Missing values",
        "Duplicate rows",
        "Duplicate transaction IDs",
        "Invalid fraud labels",
        "Amount outliers",
        "Training rows",
        "Testing rows"
    ],

    "Result": [
        len(df),
        len(df.columns),
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum()),
        int(df["trans_num"].duplicated().sum()),
        int(
            (~df["is_fraud"].isin([0, 1])).sum()
        ),
        int(amount_outliers.sum()),
        len(X_train),
        len(X_test)
    ]
})

display(quality_log)


,Check,Result
0,Dataset rows,339607
1,Dataset columns,21
2,Missing values,0
3,Duplicate rows,0
4,Duplicate transaction IDs,0
5,Invalid fraud labels,0
6,Amount outliers,18043
7,Training rows,271685
8,Testing rows,67922


In [51]:
#Save audit log
quality_log.to_csv(
    os.path.join(
        processed_dir,
        "data_quality_audit_log.csv"
    ),
    index=False
)

print("Data quality audit log saved successfully!")

Data quality audit log saved successfully!


In [53]:
#Final file check
print("========== FINAL FILE CHECK ==========")

for file in os.listdir(processed_dir):
    print("✓", file)

print("\nFolder location:")
print(os.path.abspath(processed_dir))

========== FINAL FILE CHECK ==========
✓ data_quality_audit_log.csv
✓ processed_credit_card_fraud.csv
✓ X_test.csv
✓ X_test_processed.csv
✓ X_train.csv
✓ X_train_processed.csv
✓ y_test.csv
✓ y_train.csv

Folder location:
C:\Users\AIRA\data\processed
